# CAD in the authors' environment (Python 3.8, PyTorch 1.9.0+cu111)
Question: does the 16–18 point reproduction gap close with the authors' software stack? Same config, same seeds, same three arms as E3, run in a separate Python 3.8 environment.

**Inputs:** your CAD dataset (Add Input).
**Settings:** **Accelerator: GPU T4 x2**, Internet On → **Save Version → Save & Run All**. About 1 h per run; the loop stops starting new runs after ~11 h, so CHASE and fixed (6 runs) finish first.
**Send back:** `cad_torch19_results.csv`.

## 1. Setup

In [ ]:
import os, sys, re, glob, json, time, zipfile, tarfile, pickle, shutil, subprocess
def sh(cmd, cwd=None):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

WORK = '/kaggle/working'
REPO = WORK + '/CHASE/NeurIPS24'
if not os.path.isdir(REPO):
    sh('git clone -q --depth 1 https://github.com/Necolizer/CHASE.git', cwd=WORK)
sh('pip install -q dotmap einops pyyaml')
import torch, yaml
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Turn on GPU in Session options'


## 2. CAD data

In [ ]:
# CAD skeletons from the attached CAD dataset (Add Input)
DATA_ROOT = '/tmp/cad'
os.makedirs(DATA_ROOT, exist_ok=True)
hits = glob.glob(DATA_ROOT + '/**/tracks_normalized.pkl', recursive=True)
if not hits:
    src_hits = glob.glob('/kaggle/input/**/tracks_normalized.pkl', recursive=True)
    if src_hits:
        shutil.copytree(os.path.dirname(src_hits[0]), os.path.join(DATA_ROOT, 'collective'),
                        ignore=shutil.ignore_patterns('videos'))
    else:
        arcs = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
                if os.path.isfile(p) and not p.endswith('.pt') and (zipfile.is_zipfile(p) or tarfile.is_tarfile(p))]
        arcs = [a for a in arcs if 'collective' in a.lower() or 'cad' in a.lower()] or arcs
        assert arcs, 'Attach the CAD dataset via Add Input'
        a = arcs[0]
        if zipfile.is_zipfile(a):
            with zipfile.ZipFile(a) as z:
                z.extractall(DATA_ROOT, [m for m in z.namelist() if 'videos' not in m.split('/')])
        else:
            with tarfile.open(a) as t:
                t.extractall(DATA_ROOT, [m for m in t.getmembers() if 'videos' not in m.name.split('/')])
    hits = glob.glob(DATA_ROOT + '/**/tracks_normalized.pkl', recursive=True)
assert hits, 'tracks_normalized.pkl not found'
CAD_DIR = os.path.dirname(hits[0])
print('CAD_DIR =', CAD_DIR)


## 3. Authors' environment (Python 3.8 + torch 1.9)

In [ ]:
# Authors' stack: Python 3.8, torch 1.9.0+cu111, numpy 1.19.5 (requirements.txt), in a separate venv.
PY = '/tmp/py38/bin/python'
NEED = 'numpy==1.19.5 pandas==1.3.5 scipy==1.7.3 matplotlib==3.5.3 dotmap==1.3.30 einops==0.6.1 pyyaml tqdm'
if not os.path.exists(PY):
    sh('pip install -q uv')
    try:
        sh('uv python install 3.8')
        sh('uv venv -p 3.8 /tmp/py38')
    except subprocess.CalledProcessError:          # fallback: micromamba
        sh('curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba')
        sh('/tmp/bin/micromamba create -y -q -p /tmp/py38 -c conda-forge python=3.8 pip')
    inst = f'uv pip install --python {PY}' if os.path.exists('/tmp/py38/pyvenv.cfg') else f'{PY} -m pip install -q'
    sh(f'{inst} "torch==1.9.0+cu111" "torchvision==0.10.0+cu111" -f https://download.pytorch.org/whl/torch_stable.html')
    sh(f'{inst} numpy==1.19.5 pandas==1.3.5 scipy==1.7.3 matplotlib==3.5.3 dotmap==1.3.30 einops==0.6.1 pyyaml tqdm')
sh(f'{PY} -c "import sys, torch, numpy; print(sys.version.split()[0], torch.__version__, numpy.__version__, torch.cuda.is_available(), torch.cuda.device_count(), torch.backends.cudnn.version())"')

# FIX: make sure every module imported by feeders/__init__.py is present (pandas was missing)
inst = f'uv pip install --python {PY}' if os.path.exists('/tmp/py38/pyvenv.cfg') else f'{PY} -m pip install -q'
sh(f'{inst} {NEED}')
sh(f'cd {REPO} && {PY} -c "import feeders, model, utils.loss; print(\'repo imports OK\')"')


## 4. Fixed-origin wrapper

In [ ]:
FX = r'''"""
fixed_origin.py -- baselines for the CAD "is CHASE a fixed-joint recentring?" test.
Copy to CHASE/NeurIPS24/model/Wrapper/fixed_origin.py.

mode='joint' : origin := x[:, :, t, v, m] for every sample (the point the released
               CHASE checkpoint selects in 761/761 CAD test clips: t=3, v=4, m=0).
               Padding is handled exactly as in CHASWrapper (x - p for all points),
               so the ONLY difference from CHASE is that the origin is fixed, not learned.
mode='none'  : no shift (vanilla backbone), same wrapper so eval returns logits.
"""
import torch.nn as nn


class FixedOriginWrapper(nn.Module):
    def __init__(self, backbone, mode="joint", t=3, v=4, m=0, **unused):
        super().__init__()
        assert mode in ("joint", "none")
        self.backbone, self.mode, self.t, self.v, self.m = backbone, mode, t, v, m

    def forward(self, x):
        if self.mode == "joint":
            p = x[:, :, self.t, self.v, self.m]                     # (N, C)
            x = x - p[:, :, None, None, None]
        out = self.backbone(x)
        if not self.training and isinstance(out, tuple):
            return out[0]
        return out
'''
open(REPO + '/model/Wrapper/fixed_origin.py', 'w').write(FX)
init = REPO + '/model/Wrapper/__init__.py'
if 'fixed_origin' not in open(init).read():
    open(init, 'a').write('\nfrom . import fixed_origin\n')
print(open(init).read())


## 5. Configs

In [ ]:
NGPU = torch.cuda.device_count()
assert NGPU >= 2, 'Use Accelerator: GPU T4 x2 (paper batch 256 does not fit one T4)'
base = yaml.safe_load(open(REPO + '/config/CTRGCN/cad/cad_chase.yaml'))
for k in ('train_feeder_args', 'test_feeder_args'):
    base[k]['dataset_dir'] = CAD_DIR
base.update(device=[0, 1], cuda_visible_device='0,1', num_worker=4, run_mode='train')
def arm_cfg(arm):
    c = json.loads(json.dumps(base))
    if arm in ('fixed', 'vanilla'):
        c['wrapper'] = 'model.Wrapper.fixed_origin.FixedOriginWrapper'
        c['wrapper_args'] = dict(mode='joint' if arm == 'fixed' else 'none', t=3, v=4, m=0)
        c['loss'] = 'LSCE_GROUP'
        c['loss_args'] = dict(smoothing=base['loss_args']['smoothing'], temperature=base['loss_args']['temperature'])
    yaml.safe_dump(c, open(f'{REPO}/cad_{arm}.yaml', 'w'))
for arm in ('chase', 'fixed', 'vanilla'):
    arm_cfg(arm)
print('configs written; batch', base['batch_size'], 'lr', base['base_lr'])


## 6. Train

In [ ]:
# CHASE first (the reproduction question), paired with fixed; vanilla last.
SEEDS = [1, 2, 3]
PLAN = [(a, s) for s in SEEDS for a in ('chase', 'fixed')] + [('vanilla', s) for s in SEEDS]
RES = WORK + '/cad_torch19_results.csv'
done = set()
if os.path.exists(RES):
    for line in open(RES).read().splitlines()[1:]:
        a, s = line.split(',')[:2]; done.add((a, int(s)))
else:
    open(RES, 'w').write('arm,seed,best_acc,last_acc,minutes,env\n')
os.makedirs(WORK + '/runs19', exist_ok=True)
pat = re.compile(r'top1: ([\d.]+)%, best_acc: ([\d.]+)%')
T_START, BUDGET_MIN = time.time(), 11 * 60          # stop starting new runs after ~11 h
for arm, seed in PLAN:
    if (arm, seed) in done:
        continue
    if (time.time() - T_START) / 60 > BUDGET_MIN - 60:
        print('time budget reached, stopping'); break
    wd = f'{WORK}/runs19/cad_{arm}_s{seed}'
    shutil.rmtree(wd, ignore_errors=True)
    t0 = time.time()
    sh(f'{PY} main_group.py --config cad_{arm}.yaml --seed {seed} --work_dir {wd} > {wd}.out 2>&1 || (tail -40 {wd}.out; false)', cwd=REPO)
    m = pat.findall(open(wd + '/log.txt').read())
    last, best = float(m[-1][0]), float(m[-1][1])
    mins = (time.time() - t0) / 60
    open(RES, 'a').write(f'{arm},{seed},{best},{last},{mins:.1f},torch1.9\n')
    print(f'>>> {arm} seed {seed}: best {best:.2f}  last {last:.2f}  ({mins:.1f} min)', flush=True)


## 7. Summary

In [ ]:
import pandas as pd
df = pd.read_csv(RES); print(df.to_string(index=False))
print(df.groupby('arm')[['best_acc', 'last_acc']].agg(['mean', 'std', 'count']).round(2))
print('\nPaper: vanilla 80.45 +- 2.29, CHASE 89.61 +- 0.20.  Previous env (torch 2.x): vanilla 64.0, CHASE 71.4, fixed 74.8')
print('Send back: /kaggle/working/cad_torch19_results.csv')
